# Tahap 4 — Case Solution Reuse (Prediksi Solusi)

Dari top-k kasus termirip, sistem mengambil elemen putusan sebagai solusi atas kasus baru.

Dua algoritma prediksi:
1. **Majority Vote** — pilih label_pasal yang paling sering muncul di top-k
2. **Weighted Vote** — bobot = similarity^5 (kasus paling mirip mendominasi)
3. **Weighted Average** — rata-rata tertimbang `vonis_bulan` berdasarkan similarity

**Input**: `data/eval/queries.json`, `data/processed/cases.json`, model terlatih

**Output**: `data/results/predictions.csv`

## 4.1 Import Library

In [ ]:
import json
import pickle
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")

## 4.2 Load Data dan Model

In [ ]:
DATA_DIR   = Path("../data/processed")
EVAL_DIR   = Path("../data/eval")
MODEL_DIR  = Path("../models")
RESULT_DIR = Path("../data/results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "cases.json", encoding="utf-8") as f:
    cases = json.load(f)
case_map = {c["case_id"]: c for c in cases}

with open(MODEL_DIR / "tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

all_embeddings = np.load(MODEL_DIR / "indobert_embeddings.npy")
all_vectors    = vectorizer.transform([c["text_full"] for c in cases])

print(f"Loaded {len(cases)} kasus")
print(f"TF-IDF vectors: {all_vectors.shape}")
print(f"IndoBERT embeddings: {all_embeddings.shape}")

## 4.3 Load IndoBERT (CPU)

In [ ]:
device     = torch.device("cpu")
MODEL_NAME = "indobenchmark/indobert-base-p1"
print(f"Loading {MODEL_NAME} di CPU...")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()
print("Model berhasil di-load")

## 4.4 Fungsi Retrieval dan Prediksi

In [ ]:
def get_indobert_embedding(text, max_length=512):
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=max_length, padding=True
    )
    with torch.no_grad():
        outputs = bert_model(**inputs)
    mask = inputs["attention_mask"].unsqueeze(-1).float()
    emb  = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
    return emb.numpy()[0]


def retrieve_tfidf(query_text, k=5):
    q_vec = vectorizer.transform([query_text])
    sims  = cosine_similarity(q_vec, all_vectors)[0]
    top_k = np.argsort(sims)[::-1][:k]
    return [(cases[i]["case_id"], sims[i]) for i in top_k]


def retrieve_bert(query_text, k=5):
    q_emb = get_indobert_embedding(query_text).reshape(1, -1)
    sims  = cosine_similarity(q_emb, all_embeddings)[0]
    top_k = np.argsort(sims)[::-1][:k]
    return [(cases[i]["case_id"], sims[i]) for i in top_k]


def majority_vote(top_k_with_sim, key):
    values = [case_map[cid].get(key, "") for cid, _ in top_k_with_sim]
    values = [v for v in values if v]
    if not values:
        return "tidak_diketahui"
    return Counter(values).most_common(1)[0][0]


def weighted_vote(top_k_with_sim, key):
    scores = {}
    for cid, sim in top_k_with_sim:
        val = case_map[cid].get(key, "")
        if val:
            scores[val] = scores.get(val, 0.0) + (sim ** 5)
    if not scores:
        return "tidak_diketahui"
    return max(scores, key=scores.get)


def predict_vonis_bulan(top_k_with_sim, method="weighted"):
    if not top_k_with_sim:
        return 0
    if method == "majority":
        vals = [case_map[cid].get("vonis_bulan", 0) for cid, _ in top_k_with_sim]
        return int(round(sum(vals) / len(vals)))
    total_weight = sum(sim ** 5 for _, sim in top_k_with_sim)
    if total_weight == 0:
        return 0
    weighted = sum(case_map[cid].get("vonis_bulan", 0) * (sim ** 5) for cid, sim in top_k_with_sim)
    return int(round(weighted / total_weight))

## 4.5 Jalankan Prediksi pada Query Evaluasi

In [ ]:
with open(EVAL_DIR / "queries.json", encoding="utf-8") as f:
    queries = json.load(f)

rows = []
for q in queries:
    qid      = q["query_id"]
    gt_pasal = q["ground_truth_label_pasal"]
    gt_vonis = q["ground_truth_vonis_bulan"]
    qtext    = q["query_text"]

    tfidf_top_k = retrieve_tfidf(qtext, k=5)
    bert_top_k  = retrieve_bert(qtext, k=5)

    rows.append({
        "query_id":             qid,
        "ground_truth_pasal":   gt_pasal,
        "ground_truth_vonis":   gt_vonis,
        "tfidf_mv_pasal":       majority_vote(tfidf_top_k, "label_pasal"),
        "tfidf_mv_vonis":       predict_vonis_bulan(tfidf_top_k, "majority"),
        "tfidf_wv_pasal":       weighted_vote(tfidf_top_k, "label_pasal"),
        "tfidf_wv_vonis":       predict_vonis_bulan(tfidf_top_k, "weighted"),
        "bert_mv_pasal":        majority_vote(bert_top_k, "label_pasal"),
        "bert_mv_vonis":        predict_vonis_bulan(bert_top_k, "majority"),
        "bert_wv_pasal":        weighted_vote(bert_top_k, "label_pasal"),
        "bert_wv_vonis":        predict_vonis_bulan(bert_top_k, "weighted"),
        "top5_tfidf_case_ids":  "|".join(c for c, _ in tfidf_top_k),
        "top5_bert_case_ids":   "|".join(c for c, _ in bert_top_k),
    })

    r = rows[-1]
    print(f"  {qid}: GT pasal={gt_pasal}, GT vonis={gt_vonis} bln")
    print(f"         TF-IDF MV: pasal={r['tfidf_mv_pasal']}, vonis={r['tfidf_mv_vonis']} bln")
    print(f"         TF-IDF WV: pasal={r['tfidf_wv_pasal']}, vonis={r['tfidf_wv_vonis']} bln")
    print(f"         BERT   MV: pasal={r['bert_mv_pasal']}, vonis={r['bert_mv_vonis']} bln")
    print(f"         BERT   WV: pasal={r['bert_wv_pasal']}, vonis={r['bert_wv_vonis']} bln")
    print()

df = pd.DataFrame(rows)
df.to_csv(RESULT_DIR / "predictions.csv", index=False)
print(f"Predictions disimpan: {RESULT_DIR / 'predictions.csv'}")
df